# AgentDebuggerEnv — GRPO Training

**Training Qwen2.5-Coder-7B-Instruct on structured hypothesis-driven debugging**

- **Algorithm:** GRPO (same as DeepSeek-R1) via HuggingFace TRL
- **Dataset:** 90 hand-validated bugs across 3 difficulty tiers
- **Curriculum:** Tier 1 (steps 0–150) → Tier 1+2 (150–350) → All tiers (350–500)
- **Model:** Qwen2.5-Coder-7B-Instruct + LoRA (float16/bfloat16, no quantization)

> **Requirements:** GPU runtime. In Colab: Runtime → Change runtime type → **A100**.

In [ ]:
# Verify GPU is available
import subprocess, sys
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → GPU (A100 recommended)")
print(result.stdout[:600])

In [ ]:
# Clone the environment repository
!git clone https://huggingface.co/spaces/shashaank0707/AgentDebugger-env agentdebugger
%cd agentdebugger

In [ ]:
# Install CUDA-enabled PyTorch first (must precede all other imports)
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# Install training dependencies
!pip install -q \
    wandb==0.18.7 \
    datasets==3.0.2 \
    transformers==4.48.3 \
    accelerate==1.0.1 \
    "trl==0.15.2" \
    peft==0.13.2

import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:            {props.name}")
    print(f"VRAM:           {props.total_memory / 1e9:.1f} GB")

In [ ]:
import os

# Weights & Biases — get a free API key at https://wandb.ai
WANDB_API_KEY = ""  # @param {type:"string"}
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb; wandb.login(key=WANDB_API_KEY)
    print("W&B login successful — training curves will be logged")
else:
    print("No W&B key — set WANDB_API_KEY above to get loss/reward plots")

# Hugging Face token — needed to push the final model
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login; login(token=HF_TOKEN)
    print("HF login successful — trained model will be pushed to Hub")

## Step 1 — Sanity Check (10 steps, ~2 min)

Runs 10 training steps to verify GPU, dependencies, and reward function all work before the full run.

In [ ]:
!python training/train_grpo.py --test

## Step 2 — Full Training (500 steps, ~45 min on A100)

Runs the complete curriculum:
- **Steps 0–150:** Tier 1 only (easy bugs — off-by-one, simple logic)
- **Steps 150–350:** Tier 1 + Tier 2 (adds red-herring auth bugs)
- **Steps 350–500:** All tiers (adds concurrency race conditions)

Checkpoints saved every 50 steps. Final model pushed to HF Hub if `HF_TOKEN` is set.

In [ ]:
!python training/train_grpo.py

## Results — Baseline vs Trained

In [ ]:
import json, os

baseline, final = None, None

if os.path.exists("baseline_results.json"):
    with open("baseline_results.json") as f:
        baseline = json.load(f)
    print(f"Baseline  | solve_rate: {baseline['solve_rate']:.1%} | avg_reward: {baseline['avg_reward']:.3f}")

if os.path.exists("final_results.json"):
    with open("final_results.json") as f:
        final = json.load(f)
    print(f"Trained   | solve_rate: {final['solve_rate']:.1%} | avg_reward: {final['avg_reward']:.3f}")
    if baseline:
        delta = final['avg_reward'] - baseline['avg_reward']
        print(f"\nImprovement: {delta:+.3f} ({delta / baseline['avg_reward'] * 100:+.1f}% relative)")
else:
    print("final_results.json not written yet — run training first")